### 0. Composition
My knowledge on transceivers is a bit limited. I am aiming for a very simple scheme, where tx consists of a simple MZM for transmitting signal through intensity modulation.

I am assuming the laser to be part of a different chip, and have thus optical input and output from north to couple to an MZM.

In [ ]:
import gdsfactory as gf
from gdsfactory.typings import Layer

# @gf.cell
# def transceiver():

c = gf.Component()

mzm = c << gf.components.mzis.mzm()

mzm.move((10,-500))

### 1. Optical coupling and connections for tx part.
I added a layer to etch the oxide for coupling.

In [ ]:
tx_couplers = c << gf.components.grating_coupler_array(grating_coupler='grating_coupler_elliptical', pitch=127, n=3, port_name='o1', rotation=90, with_loopback=False, cross_section='strip', straight_to_grating_spacing=10, centered=True, bend='bend_euler', mirror_grating_coupler=False).copy()

# Adding clad openings for the grating couplers
GRATING_OPEN: Layer = (120, 0)  # since I believe there is no such layer in the generic PDK
clad_opening_width = 30
y = 30.6
for i in range(3):
    c.add_polygon([[tx_couplers[i].x - clad_opening_width/2, y-clad_opening_width/2],
                [tx_couplers[i].x - clad_opening_width/2, y+clad_opening_width/2],
                [tx_couplers[i].x + clad_opening_width/2, y+clad_opening_width/2],
                [tx_couplers[i].x + clad_opening_width/2, y-clad_opening_width/2]], layer=(120,0))

route1 = gf.routing.route_bundle(
    c,
    [mzm["o1"]],
    [tx_couplers["o0"]],
    cross_section="strip"
)

route2 = gf.routing.route_astar(
    c,
    mzm["o2"],
    tx_couplers["o1"],
    cross_section="strip",
    bend="bend_euler"
)

### 2. Building a depletion phase shifter in each arm.
In order to do that I am aiming at a cross section of:

| 90nm PPP | 90nm P | 220nm P | 220nm N | 90nm N | 90nm NPP |

similar to Figure 4 from https://www.degruyterbrill.com/document/doi/10.1515/joc-2019-0259/html?srsltid=AfmBOoo4Bg-NbznLMUNL7f1TKiswAupIGJ76pnWnfuz3mBUfDbNE0RSj

Since there is a gap between the 220nm Si wg and the slabs on each side, I am covering with a 90nm slab.

In [ ]:
ps_length = 180
doped_region_width = 10
wg_width = 0.5
# this is the gap between the 220nm wg and the slab
wg_slab_gap = 0.75
# this is the gap between the waveguides of the top and bottom arm of the mzm
wg_wg_gap = 50.25

# Add slab_90s at the gaps between the waveguides and the regions under the electrodes
# Top arm, above waveguide
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2]], layer=(3,0))

# Top arm, below waveguide
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2]], layer=(3,0))

# Bottom arm, above waveguide
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2]], layer=(3,0))

# Bottom arm, below waveguide
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2]], layer=(3,0))

# Assign doped regions
# Top arm, heavily-doped slab above waveguide. PPP = (25,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap + doped_region_width],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap + doped_region_width],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap]], layer=(25,0))

# Top arm, p-type doping on adjacent upper slab and half waveguide. P = (21,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2]], layer=(21,0))

# Top arm, n-type doping on adjacent lower slab and half waveguide. N = (20,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2]], layer=(20,0))

# Top arm, heavily-doped slab below waveguide. NPP = (24,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x - ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap - doped_region_width],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap - doped_region_width],
    [mzm.x + ps_length/2, mzm.y + wg_wg_gap/2 - wg_width/2 - wg_slab_gap]], layer=(24,0))


# Bottom arm, heavily-doped slab above waveguide. PPP = (25,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap + doped_region_width],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap + doped_region_width],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap]], layer=(25,0))

# Bottom arm, p-type doping on adjacent upper slab and half waveguide. P = (21,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 + wg_width/2 + wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2]], layer=(21,0))

# Bottom arm, n-type doping on adjacent lower slab and half waveguide. N = (20,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2]], layer=(20,0))

# Bottom arm, heavily-doped slab below waveguide. NPP = (24,0)
c.add_polygon([
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap],
    [mzm.x - ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap - doped_region_width],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap - doped_region_width],
    [mzm.x + ps_length/2, mzm.y - wg_wg_gap/2 - wg_width/2 - wg_slab_gap]], layer=(24,0))

### 4. MZM pads for a lumped element configuration
The idea is to have the RF signals to the east of the die.

In [ ]:
tx_pads = c << gf.components.pad_array(port_orientation=180, columns=1, rows=4)
tx_pads.movex(1000)
tx_pads.y = mzm.y

# Adding PADOPEN layer ontop of M3 pads
for pad in tx_pads:
    c.add_polygon([[pad.x, pad.y-pad.width/2],
                [pad.x, pad.y+pad.width/2],
                [pad.x + pad.width, pad.y+pad.width/2],
                [pad.x + pad.width, pad.y-pad.width/2]], layer=(46,0))

### 4. Make rf transition for MZM's electrodes
I am arbitrarily using sinusoidal transition between the wide and narrow cross sections, to smoothen the impedance transition.
Of course in reality this would require RF simulations.

In [ ]:
mzm_width = mzm.ports["e9"].width
mzm_layer = mzm.ports["e9"].layer

port_mapping = [["e9", "e41"], ["e10", "e31"], ["e11", "e21"], ["e12", "e11"]]  # List of corresponding [mzm_port, tx_pad_port]

for mzm_port, tx_pad_port in port_mapping:

    mzm_xsection = gf.CrossSection(sections=[gf.Section(width=mzm_width, layer=mzm_layer, name="rf_connection", offset=-mzm.ports[mzm_port].y)])

    tx_pads_width = tx_pads.ports[tx_pad_port].width

    tx_pads_xsection = gf.CrossSection(sections=[gf.Section(width=tx_pads_width, layer=mzm_layer, name="rf_connection", offset=-tx_pads.ports[tx_pad_port].y)])

    # Create the transitional cross-section.
    Xtrans = gf.path.transition(cross_section1=mzm_xsection, cross_section2=tx_pads_xsection, width_type="sine")

    # Create a Path for the transitional cross-section to follow.
    P3 = gf.path.straight(length=tx_pads.ports[tx_pad_port].x - mzm.ports[mzm_port].x, npoints=100)

    # Use the transitional cross-section to create a component.
    straight_transition = c << gf.path.extrude_transition(P3, Xtrans)

    straight_transition.movex(mzm.ports[mzm_port].x)

### 5. Add receiver part
For rx I am just adding a detector, along with its optical input and RF output pads.

In [ ]:
detector = c << gf.components.ge_detector_straight_si_contacts(length=40, cross_section='pn_ge_detector_si_contacts', via_stack='via_stack_slab_m3', via_stack_width=10, via_stack_spacing=5, via_stack_offset=0, taper_length=20, taper_width=0.8, taper_cros_section='strip').copy()

detector.move((700, -1400))

route3 = gf.routing.route_astar(
    c,
    port2=detector["o1"],
    port1=tx_couplers["o2"],
    cross_section="strip",
    bend="bend_euler",
    avoid_layers=(1,0)
)

rx_pads = c << gf.components.pad_array(port_orientation=180, columns=1, rows=2)
rx_pads.movex(1000)
rx_pads.y = detector.y

detector_width = detector.ports["top_e3"].width

port_mapping = [["top_e3", "e21"], ["bot_e3", "e11"]]   # List of corresponding [detector_port, rx_pad_port]

for detector_port, rx_pad_port in port_mapping:

    detector_xsection = gf.CrossSection(sections=[gf.Section(width=detector_width, layer=mzm_layer, name="rf_connection", offset=-detector.ports[detector_port].y)])

    rx_pads_width = rx_pads.ports[rx_pad_port].width

    # Adding PADOPEN layer ontop of M3 pads
    for pad in rx_pads:
        c.add_polygon([[pad.x, pad.y-pad.width/2],
            [pad.x, pad.y+pad.width/2],
            [pad.x + pad.width, pad.y+pad.width/2],
            [pad.x + pad.width, pad.y-pad.width/2]], layer=(46,0))

    rx_pads_xsection = gf.CrossSection(sections=[gf.Section(width=rx_pads_width, layer=mzm_layer, name="rf_connection", offset=-rx_pads.ports[rx_pad_port].y)])


    # Create the transitional cross-section.
    Xtrans = gf.path.transition(cross_section1=detector_xsection, cross_section2=rx_pads_xsection, width_type="sine")

    # Create a Path for the transitional cross-section to follow.
    P3 = gf.path.straight(length=rx_pads.ports[rx_pad_port].x - detector.ports[detector_port].x, npoints=100)

    # Use the transitional cross-section to create a component.
    straight_transition = c << gf.path.extrude_transition(P3, Xtrans)

    straight_transition.movex(detector.ports[detector_port].x)

In [ ]:
c.draw_ports()

c.show()

c.write_gds("transceiver.gds")